# Assignment #1

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.preprocessing import OneHotEncoder
enc = OneHotEncoder(handle_unknown='ignore')


### 0.0.1 Reading in the Data Table

In [2]:
toyota = pd.read_csv('RAV4-142-Spring2021.csv')
toyota


,MonthNumeric,MonthFactor,Year,RAV4Sales,Unemployment,RAV4Queries,CPIAll,CPIEnergy
0,1,January,2011,11196,9.1,29,221.187,229.258
1,2,February,2011,12562,9.0,29,221.898,232.068
2,3,March,2011,16082,9.0,29,223.046,240.079
3,4,April,2011,15586,9.1,27,224.093,247.977
4,5,May,2011,8624,9.0,28,224.806,250.744
...,...,...,...,...,...,...,...,...
115,8,August,2020,39239,8.4,94,259.681,194.430
116,9,September,2020,43652,7.8,88,260.209,195.990
117,10,October,2020,40717,6.9,89,260.325,196.269
118,11,November,2020,40250,6.7,79,260.817,197.112


### 0.0.2 Some Short EDA

In [3]:
toyota.info()

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   MonthNumeric  120 non-null    int64  
 1   MonthFactor   120 non-null    str    
 2   Year          120 non-null    int64  
 3   RAV4Sales     120 non-null    int64  
 4   Unemployment  120 non-null    float64
 5   RAV4Queries   120 non-null    int64  
 6   CPIAll        120 non-null    float64
 7   CPIEnergy     120 non-null    float64
dtypes: float64(3), int64(4), str(1)
memory usage: 7.6 KB


In [4]:
# toyota.corr()

## Question 2a)

In [5]:
# Split data into training and test data
toyota_train = toyota[toyota['Year'] <= 2016]
toyota_test = toyota[toyota['Year'] > 2016]

Focus on 4 variables: *Unemployment, RAV4Queries, CPIEnergy, CPIAll*

We want to predict *RAV4Sales*

In [6]:
ols = smf.ols(formula = 'RAV4Sales ~ Unemployment + RAV4Queries + CPIEnergy + CPIAll', data=toyota_train)

cols = ['Unemployment', 'RAV4Queries', 'CPIEnergy', 'CPIAll']

In [7]:
ols

In [8]:
cols

['Unemployment', 'RAV4Queries', 'CPIEnergy', 'CPIAll']

In [9]:
# Model 1
model_1 = ols.fit()
print(model_1.summary())

                            OLS Regression Results                            
Dep. Variable:              RAV4Sales   R-squared:                       0.810
Model:                            OLS   Adj. R-squared:                  0.799
Method:                 Least Squares   F-statistic:                     71.42
Date:                Fri, 11 Sep 2026   Prob (F-statistic):           1.93e-23
Time:                        09:34:01   Log-Likelihood:                -683.31
No. Observations:                  72   AIC:                             1377.
Df Residuals:                      67   BIC:                             1388.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     1.961e+04   9.36e+04      0.210   

### Variance Inflation Factor (VIF)

A measure of the amount of multicollinearity in a set of multiple regression variables.

In [10]:
# calculate Variance Inflation Factor for each explanatory variable
from statsmodels.stats.outliers_influence import variance_inflation_factor

def VIF(df, columns):
    # sm.add_constant --> Add a column of ones to an array.
    values = sm.add_constant(df[columns]).values
    
    num_columns = len(columns) + 1
    
    vif = [variance_inflation_factor(values, i) for i in range(num_columns)]
    
    return pd.Series(vif[1:], index=columns)

In [11]:
VIF(toyota_train, cols)

Unemployment    37.437684
RAV4Queries      6.231404
CPIEnergy        7.220536
CPIAll          28.216088
dtype: float64

In [12]:
# Remove Unemployment because of its high VIF
model_2 = smf.ols(formula = 'RAV4Sales ~ RAV4Queries + CPIEnergy + CPIAll', data=toyota_train).fit()
print(model_2.summary())

                            OLS Regression Results                            
Dep. Variable:              RAV4Sales   R-squared:                       0.801
Model:                            OLS   Adj. R-squared:                  0.792
Method:                 Least Squares   F-statistic:                     91.21
Date:                Fri, 11 Sep 2026   Prob (F-statistic):           8.73e-24
Time:                        09:34:01   Log-Likelihood:                -684.99
No. Observations:                  72   AIC:                             1378.
Df Residuals:                      68   BIC:                             1387.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept   -1.379e+05   3.21e+04     -4.290      

In [13]:
cols = ['RAV4Queries', 'CPIEnergy', 'CPIAll']
VIF(toyota_train, cols)

RAV4Queries    5.952056
CPIEnergy      2.610675
CPIAll         3.876882
dtype: float64

In [14]:
# Remove RAV4Queries for its high VIF
model_3 = smf.ols(formula = 'RAV4Sales ~ CPIEnergy + CPIAll', data=toyota_train).fit()
print(model_3.summary())

                            OLS Regression Results                            
Dep. Variable:              RAV4Sales   R-squared:                       0.790
Model:                            OLS   Adj. R-squared:                  0.784
Method:                 Least Squares   F-statistic:                     130.1
Date:                Fri, 11 Sep 2026   Prob (F-statistic):           3.85e-24
Time:                        09:34:01   Log-Likelihood:                -686.84
No. Observations:                  72   AIC:                             1380.
Df Residuals:                      69   BIC:                             1387.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept  -1.729e+05   2.68e+04     -6.447      0.0

In [15]:
cols = ['CPIEnergy', 'CPIAll']
VIF(toyota_train, cols)

CPIEnergy    1.67804
CPIAll       1.67804
dtype: float64

All the p-values are now 0 and all the VIFs are reasonably low.

### 1.0.1 Explanation (ii, iii, iv)


When I ran model_1, the one with all 4 of the independent variables, I got an R^2 of 0.810. The
signs of the coefficients made sense to me. Unemployment was negative which makes sense, since
unemployed people tend to have little to no income. This means they will be less likely to purchase
a Toyota RAV4. CPIAll was positive which makes sense, since it corresponds to the consumer
price index-a measure of how much each household essentially spent. Thus it makes sense that
this coefficient would increase with Toyota RAV4 sales. The variable RAV4Queries is also positive,
which makes sense, since it corresponds to the normalized approximation of the number of Google
searches with the query ‘Toyota RAV4’. Naturally the more buzz a product has, the more likely
it is to be sought after by consumers. Lastly, the variable CPIEnergy has a negative coefficient,
which makes sense since it pertains to the monthly CPI for the energy sector. Some consumers
may be spending more of their money on paying off electricity bills or buying gas for their current
cars, hence why they may not be in the market for a new vehicle, like the Toyota RAV4. Some of
the coefficients had high p-values, so I assumed that there could be some multicollinearity.


To check for multicollinearity, I used the VIF approach to determine which independent variables
to remove. All of the VIFs were higher than 10 for the initial model. The variable with the highest
VIF was Unemployment, so I removed it first and reran the model. The VIF values had decreased
but there were still relatively high. RAV4Queries had the highest VIF value so I removed it next.


I was down to 2 independent variables in my third model. When I ran the model, I noted that all
of the p-values had gone to zero, meaning that my remaining variables, CPIEnergy and CPIAll,
were statistically significant. The signs didn’t change either which is good. I checked the VIF values
to make sure there was no multicollinearity. All of the VIF values were under 2 so I decided that
this was a good enough model to stop at.


### 1.0.2 Interpretation of Coefficients in Most Recent Model (i)


**Final Model : RAV4Sales = -172900.0 - 95.5914(CPIEnergy) + 920.5497(CPIAll)** 

• The final model has 1 intercept and only 2 independent variables: CPIEnergy and CPIAll.

• The intercept has a value of -172900.0, meaning if CPIEnergy and CPIAll were set to 0, the
default number of sales would have been -172900.

• CPIEnergy has a coefficient of -95.5914. This means for every addiitonal unit increase in the
monthly consumer price index in the energy sector, RAV4Sales is predicted to decrease by
-95.5914 units.

• CPIAll has a coefficient of 920.5497. This means for every additional unit increase in the
consumer price index for all products, RAV4Sales is predicted to increase by 920.5497 units.

## 2 Question 2b)

In [16]:
# Using seasonality  --> MonthFactor
model_4 = smf.ols(formula = 'RAV4Sales ~ Unemployment + RAV4Queries + CPIEnergy + CPIAll + MonthFactor', data=toyota_train).fit()
print(model_4.summary())

                            OLS Regression Results                            
Dep. Variable:              RAV4Sales   R-squared:                       0.884
Model:                            OLS   Adj. R-squared:                  0.853
Method:                 Least Squares   F-statistic:                     28.51
Date:                Fri, 11 Sep 2026   Prob (F-statistic):           8.55e-21
Time:                        09:34:01   Log-Likelihood:                -665.48
No. Observations:                  72   AIC:                             1363.
Df Residuals:                      56   BIC:                             1399.
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

### 2.0.1 i)

**Model:**

*RAV4Sales = 7.754e+04 - 3.687.3648(Unemployment) + 228.4423(RAV4Queries) + 1.1895(CPIEnergy) -175.4430(CPIAll)*  

*+ 2422.0989(MonthFactor[T.August]) + 1885.2249(MonthFactor[T.December]) - 2922.8349(MonthFactor[T.February])*  

*-4543.8071(MonthFactor[T.January]) - 193.7079(MonthFactor[T.July]) - 1426.1733(MonthFactor[T.June])*  

*+466.8540(MonthFactor[T.March]) + 2010.0694(MonthFactor[T.May]) - 1540.1770(MonthFactor[T.November])* 

*-1808.4239(MonthFactor[T.October]) - 1879.2594(MonthFactor[T. September])*

### 2.1 Interpretation of Coefficients (Each of the MonthFactor Dummy Variables)


- The January dummy variable has a coefficient of -4543.8071. That means on average during January, 4543.8071 less units of the Toyota RAV4 are sold.


- The February dummy variable has a coefficient of -2922.8349. That means on average during February, 2922.8349 less units of the Toyota RAV4 are sold.


- The March dummy variable has a coefficient of 466.8540. That means on average during March, 466.8540 more units of the Toyota RAV4 are sold.


- The April dummy variable has been removed to avoid the Dummy Variable Trap.


- The May dummy variable has a coefficient of 2010.0694. That means on average during May, 2010.0694 more units of the Toyota RAV4 are sold.


- The June dummy variable has a coefficient of -1426.1733. That means on average during June, 1426.1733 less units of the Toyota RAV4 are sold.


- The July dummy variable has a coefficient of -193.7079. That means on average during July, 193.7079 less units of the Toyota RAV4 are sold.


- The August dummy variable has a coefficient of 2422.0989. That means on average during August, 2422.0989 more units of the Toyota RAV4 are sold.


- The September dummy variable has a coefficient of -1879.2594. That means on average during September, 1879.2594 less units of the Toyota RAV4 are sold.


- The October dummy variable has a coefficient of -1808.4239. That means on average during October, 1808.4239 less units of the Toyota RAV4 are sold.


- The November dummy variable has a coefficient of -1540.1770. That means on average during November, 1540.1770 less units of the Toyota RAV4 are sold.


- The December dummy variable has a coefficient of 1885.2249. That means on average during December, 1885.2249 more units of the Toyota RAV4 are sold.


### 2.1.1 ii)


### 2.1.2 R Squared


- For the new model, the new training set R^2 is 0.884.

#### 2.1.3 Which variables are significant?


• Using a p-value cutoff of 5%, only ‘Unemployment’ and the January dummy variable are
significant. Their p-values are smaller than 0.05. ‘Unemployment’ has a p-value of 0.013 and
‘January’ has a p-value of 0.008.

### 2.1.4 iii)


• I think that adding the adding the independent variable ‘MonthFactor’ did indeed improve
the model. The R^2 is quite high, with a value of 0.884, so the model performed pretty
well on the training set. It is likely that adding monthly effects helped our model perform
better as it allowed for seasonality. During certain months the Toyota RAV4 would sell a
profitable amount of units while for others, it would not. Taking seasonality into account
helped improved the performance of the model on the training set. Of course there is a
chance that we may be overfitting our model as well.


### 2.1.5 iv


• With the given data, I may actually try grouping the months into dummy variables pertaining
to whichever of the 4 seasons they correspond to. For example, I would have a winter dummy
variable that encompasses December, January, and February. There would be a total of 4
of these dummy variables: Spring, Summer, Fall, and Winter. We could then remove one of
these 4 dummy variables to avoid the dummy variable trap. This would allow for us to see
which seasons have more profitable sales and which ones don’t. This can also help make our
model less complex.


## 3 Question 2c)


• I decided to keep the same independent variables with low p-values from part a : CPIEnergy
and CPIAll.

• When including the ‘MonthFactor’ variable, ‘Unemployment’ has an extremely low p-value
in part b. So I was tempted to reinclude it into the model. Just intuitively, I feel like
unemployment is slightly relevant as to whether or not a consumer decided to purchase a new
car-in this case, a Toyota RAV4.

• Looking at the p-values for each of the month dummy variables, I noticed that a majority
of them had p-values larger than 0.05. Only January was significant with a p-value of 0.008,
so I decided to check the VIFs before deciding whether or not to remove ‘MonthFactor’ all
together.


In [17]:
model_4.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:              RAV4Sales   R-squared:                       0.884
Model:                            OLS   Adj. R-squared:                  0.853
Method:                 Least Squares   F-statistic:                     28.51
Date:                Fri, 11 Sep 2026   Prob (F-statistic):           8.55e-21
Time:                        09:34:01   Log-Likelihood:                -665.48
No. Observations:                  72   AIC:                             1363.
Df Residuals:                      56   BIC:                             1399.
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
============================================================================================
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept                 7.754e+04   8.82e+04      0.879      0.383   -9.92e+04    2.54e+05
MonthFactor[T.August]     2422.0989   1679.828      1.442      0.155    -943.001    5787.199
MonthFactor[T.December]   1885.2249   1704.333      1.106      0.273   -1528.965    5299.414
MonthFactor[T.February]  -2922.8349   1644.025     -1.778      0.081   -6216.214     370.544
MonthFactor[T.January]   -4543.8071   1648.557     -2.756      0.008   -7846.264   -1241.350
MonthFactor[T.July]       -193.7079   1687.715     -0.115      0.909   -3574.607    3187.191
MonthFactor[T.June]      -1426.1733   1666.576     -0.856      0.396   -4764.726    1912.380
MonthFactor[T.March]       466.8540   1639.891      0.285      0.777   -2818.243    3751.951
MonthFactor[T.May]        2010.0694   1640.329      1.225      0.226   -1275.904    5296.043
MonthFactor[T.November]  -1540.1770   1687.060     -0.913      0.365   -4919.765    1839.411
MonthFactor[T.October]   -1808.4239   1695.135     -1.067      0.291   -5204.188    1587.340
MonthFactor[T.September] -1879.2594   1651.944     -1.138      0.260   -5188.501    1429.982
Unemployment             -3687.3648   1437.100     -2.566      0.013   -6566.223    -808.507
RAV4Queries                228.4423    116.205      1.966      0.054      -4.343     461.228
CPIEnergy                    1.1895     40.426      0.029      0.977     -79.794      82.173
CPIAll                    -175.4430    379.958     -0.462      0.646    -936.590     585.704
==============================================================================
Omnibus:                        7.146   Durbin-Watson:                   1.287
Prob(Omnibus):                  0.028   Jarque-Bera (JB):               11.806
Skew:                           0.206   Prob(JB):                      0.00273
Kurtosis:                       4.941   Cond. No.                     8.70e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 8.7e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [18]:
# compute out-of-sample R-squared using the test set

# OSR2 --> 1 - [(SSR of regression model on test set/SSR of baseline model applied to test set)]
def OSR2(model, df_train, df_test, dependent_var):
    y_test = df_test[dependent_var]
    y_pred = model.predict(df_test)
    
    # Actual y minus predicted y
    SSE = np.sum((y_test - y_pred)**2)
    
    # Actual y minus baseline model predictions
    SST = np.sum((y_test - np.mean(df_train[dependent_var]))**2)
    
    return 1 - SSE/SST

In [19]:
# Copy of our training set that we will use
toyota_train_2 = toyota_train.copy(deep=True)
toyota_train_2

,MonthNumeric,MonthFactor,Year,RAV4Sales,Unemployment,RAV4Queries,CPIAll,CPIEnergy
0,1,January,2011,11196,9.1,29,221.187,229.258
1,2,February,2011,12562,9.0,29,221.898,232.068
2,3,March,2011,16082,9.0,29,223.046,240.079
3,4,April,2011,15586,9.1,27,224.093,247.977
4,5,May,2011,8624,9.0,28,224.806,250.744
...,...,...,...,...,...,...,...,...
67,8,August,2016,33171,4.9,59,240.595,189.718
68,9,September,2016,29438,5.0,54,241.068,192.158
69,10,October,2016,26429,4.9,52,241.641,195.541
70,11,November,2016,28116,4.7,48,241.993,195.927


In [20]:
# Use one-hot encoding so that we can check the VIFs of the MonthFactor variable
enc_df2 = pd.DataFrame(enc.fit_transform(toyota_train_2[['MonthFactor']]).toarray())
enc_df2

,0,1,2,3,4,5,6,7,8,9,10,11
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
67,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
68,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
69,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
70,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [21]:
# toyota_train2 --> A copy of the data PLUS the one-hot encoded variables. This is to calculate the VIFs
toyota_train_2 = toyota_train_2.join(enc_df2)
toyota_train_2

,MonthNumeric,MonthFactor,Year,RAV4Sales,Unemployment,RAV4Queries,CPIAll,CPIEnergy,0,1,2,3,4,5,6,7,8,9,10,11
0,1,January,2011,11196,9.1,29,221.187,229.258,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,February,2011,12562,9.0,29,221.898,232.068,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,March,2011,16082,9.0,29,223.046,240.079,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,4,April,2011,15586,9.1,27,224.093,247.977,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,May,2011,8624,9.0,28,224.806,250.744,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,8,August,2016,33171,4.9,59,240.595,189.718,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
68,9,September,2016,29438,5.0,54,241.068,192.158,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
69,10,October,2016,26429,4.9,52,241.641,195.541,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
70,11,November,2016,28116,4.7,48,241.993,195.927,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [22]:
cols = ['CPIEnergy', 'CPIAll', 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
VIF(toyota_train_2, cols)

CPIEnergy    1.750455
CPIAll       1.831353
1            1.846043
2            1.847940
3            1.839633
4            1.852171
5            1.837361
6            1.836103
7            1.834717
8            1.834210
9            1.849895
10           1.852653
11           1.848480
dtype: float64

* Seeing as all the VIF values for the MonthFactor are reasonable, I decided to include it in my final model.

In [23]:
model_5 = smf.ols(formula = 'RAV4Sales ~ CPIEnergy + CPIAll + MonthFactor', data=toyota_train).fit()
print(model_5.summary())

                            OLS Regression Results                            
Dep. Variable:              RAV4Sales   R-squared:                       0.867
Model:                            OLS   Adj. R-squared:                  0.837
Method:                 Least Squares   F-statistic:                     29.08
Date:                Fri, 11 Sep 2026   Prob (F-statistic):           1.26e-20
Time:                        09:34:02   Log-Likelihood:                -670.48
No. Observations:                  72   AIC:                             1369.
Df Residuals:                      58   BIC:                             1401.
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

In [24]:
OSR2(model_5, toyota_train, toyota_test, 'RAV4Sales')

np.float64(0.7922709030686702)

* I think that my model would be useful to Toyota in the state it is in right now. Seeing
    as I included the ‘MonthFactor’ variable, my model is now utilizing seasonality. Toyota
    is able to for example, determine during which months or seasonal periods, their RAV4 is
    sold most often. They can use this information to decide on whether or not to increase
    stock/manufacturing or decrease it for a certain month/seasonal period.
    

* R^2 is 0.867


* OSR^2 is 0.7922

## 4 Question 2d)


* Downloaded the interest over time for Toyota by using the search interest over time for the
    keyword ‘Toyota’ from between 01/01/2011 to 12/3/2020


* Numbers represent search interest relative to the highest point on the chart for the given
    region and time. A value of 100 is the peak popularity for the term. A value of 50 means
    that the term is half as popular. A score of 0 means there was not enough data for this term.

In [25]:
# Downloaded  the interest over time for the Toyota RAV4 from between 01/02/2011 to 12/03/2020
toyota_interest = pd.read_csv('ToyotaSearchInterest.csv')
toyota_interest

,Category: All categories
Month,Toyota: (United States)
2011-01,70
2011-02,74
2011-03,74
2011-04,74
...,...
2020-08,86
2020-09,85
2020-10,88
2020-11,77


### Extracting the month and years from the data.

In [26]:
toyota_interest = toyota_interest.reset_index()
toyota_interest

,index,Category: All categories
0,Month,Toyota: (United States)
1,2011-01,70
2,2011-02,74
3,2011-03,74
4,2011-04,74
...,...,...
116,2020-08,86
117,2020-09,85
118,2020-10,88
119,2020-11,77


In [27]:
toyota_interest = toyota_interest.iloc[1:]
toyota_interest

,index,Category: All categories
1,2011-01,70
2,2011-02,74
3,2011-03,74
4,2011-04,74
5,2011-05,69
...,...,...
116,2020-08,86
117,2020-09,85
118,2020-10,88
119,2020-11,77


In [28]:
toyota_interest['datetime'] = pd.to_datetime(toyota_interest['index'])

In [29]:
toyota_interest['Year'] = pd.DatetimeIndex(toyota_interest['index']).year
toyota_interest['Month'] = pd.DatetimeIndex(toyota_interest['index']).month

In [30]:
toyota_interest

,index,Category: All categories,datetime,Year,Month
1,2011-01,70,2011-01-01,2011,1
2,2011-02,74,2011-02-01,2011,2
3,2011-03,74,2011-03-01,2011,3
4,2011-04,74,2011-04-01,2011,4
5,2011-05,69,2011-05-01,2011,5
...,...,...,...,...,...
116,2020-08,86,2020-08-01,2020,8
117,2020-09,85,2020-09-01,2020,9
118,2020-10,88,2020-10-01,2020,10
119,2020-11,77,2020-11-01,2020,11


In [31]:
toyota_interest = toyota_interest.drop(columns = ['index', 'datetime'])
toyota_interest

,Category: All categories,Year,Month
1,70,2011,1
2,74,2011,2
3,74,2011,3
4,74,2011,4
5,69,2011,5
...,...,...,...
116,86,2020,8
117,85,2020,9
118,88,2020,10
119,77,2020,11


In [32]:
toyota_interest = toyota_interest.rename(columns={'Category: All categories': 'Search Interest'}).reset_index()
toyota_interest

,index,Search Interest,Year,Month
0,1,70,2011,1
1,2,74,2011,2
2,3,74,2011,3
3,4,74,2011,4
4,5,69,2011,5
...,...,...,...,...
115,116,86,2020,8
116,117,85,2020,9
117,118,88,2020,10
118,119,77,2020,11


Appending the ‘SearchInterest’ to a copy of the original toyota dataset

In [33]:
toyota2 = toyota.copy(deep = True)
toyota2

,MonthNumeric,MonthFactor,Year,RAV4Sales,Unemployment,RAV4Queries,CPIAll,CPIEnergy
0,1,January,2011,11196,9.1,29,221.187,229.258
1,2,February,2011,12562,9.0,29,221.898,232.068
2,3,March,2011,16082,9.0,29,223.046,240.079
3,4,April,2011,15586,9.1,27,224.093,247.977
4,5,May,2011,8624,9.0,28,224.806,250.744
...,...,...,...,...,...,...,...,...
115,8,August,2020,39239,8.4,94,259.681,194.430
116,9,September,2020,43652,7.8,88,260.209,195.990
117,10,October,2020,40717,6.9,89,260.325,196.269
118,11,November,2020,40250,6.7,79,260.817,197.112


In [34]:
toyota2['SearchInterest'] = toyota_interest['Search Interest'].astype('int64')
toyota2

,MonthNumeric,MonthFactor,Year,RAV4Sales,Unemployment,RAV4Queries,CPIAll,CPIEnergy,SearchInterest
0,1,January,2011,11196,9.1,29,221.187,229.258,70
1,2,February,2011,12562,9.0,29,221.898,232.068,74
2,3,March,2011,16082,9.0,29,223.046,240.079,74
3,4,April,2011,15586,9.1,27,224.093,247.977,74
4,5,May,2011,8624,9.0,28,224.806,250.744,69
...,...,...,...,...,...,...,...,...,...
115,8,August,2020,39239,8.4,94,259.681,194.430,86
116,9,September,2020,43652,7.8,88,260.209,195.990,85
117,10,October,2020,40717,6.9,89,260.325,196.269,88
118,11,November,2020,40250,6.7,79,260.817,197.112,77


In [35]:
#Split data into training and test data.
toyota_train2 = toyota2[toyota2['Year']<= 2016]
toyota_test2 = toyota2[toyota2['Year']> 2016]

In [36]:
# Running the model with my added feature
model_6 = smf.ols(formula = 'RAV4Sales ~ CPIEnergy + CPIAll + MonthFactor + SearchInterest', data=toyota_train2).fit()
print(model_6.summary())

                            OLS Regression Results                            
Dep. Variable:              RAV4Sales   R-squared:                       0.879
Model:                            OLS   Adj. R-squared:                  0.850
Method:                 Least Squares   F-statistic:                     29.68
Date:                Fri, 11 Sep 2026   Prob (F-statistic):           4.67e-21
Time:                        09:34:02   Log-Likelihood:                -666.96
No. Observations:                  72   AIC:                             1364.
Df Residuals:                      57   BIC:                             1398.
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

In [37]:
OSR2(model_6, toyota_train2, toyota_test2, 'RAV4Sales')

np.float64(0.8417367714294469)

### Does it add any predictive power?

* I decided to use use google trends to extract data regarding the search interest over time of
  the key term ‘Toyota’. I named this variable ‘SearchInterest’. I figured that interest in the
  Toyota brand in general is related to interest in their different products(RAV4). The R^2 increased from 0.867 to 0.881, so our model has improved on the training set. I believe that
    this variable does add predictive power as the OSR^2 has also increased. My original model
    had an OSR^2 of 0.7922, while this new one has an OSR^2 of 0.8089. The p-value was also
    really low at 0.012.


In [38]:
toyota_train2

,MonthNumeric,MonthFactor,Year,RAV4Sales,Unemployment,RAV4Queries,CPIAll,CPIEnergy,SearchInterest
0,1,January,2011,11196,9.1,29,221.187,229.258,70
1,2,February,2011,12562,9.0,29,221.898,232.068,74
2,3,March,2011,16082,9.0,29,223.046,240.079,74
3,4,April,2011,15586,9.1,27,224.093,247.977,74
4,5,May,2011,8624,9.0,28,224.806,250.744,69
...,...,...,...,...,...,...,...,...,...
67,8,August,2016,33171,4.9,59,240.595,189.718,87
68,9,September,2016,29438,5.0,54,241.068,192.158,83
69,10,October,2016,26429,4.9,52,241.641,195.541,78
70,11,November,2016,28116,4.7,48,241.993,195.927,74


In [39]:
toyota_train2.info()

<class 'pandas.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   MonthNumeric    72 non-null     int64  
 1   MonthFactor     72 non-null     str    
 2   Year            72 non-null     int64  
 3   RAV4Sales       72 non-null     int64  
 4   Unemployment    72 non-null     float64
 5   RAV4Queries     72 non-null     int64  
 6   CPIAll          72 non-null     float64
 7   CPIEnergy       72 non-null     float64
 8   SearchInterest  72 non-null     int64  
dtypes: float64(3), int64(5), str(1)
memory usage: 5.2 KB


In [40]:
print('finished running')

finished running
